<a href="https://colab.research.google.com/github/strongman2026/colab/blob/copilot%2Foptimize-comfyui-colab/colab_comfyui_new_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# Cell 1 / 7 : 00_env_bootstrap
# 首次/重置后运行：基础环境、ComfyUI、Manager、FRP
# ============================================================
import os, subprocess
from pathlib import Path
from google.colab import drive, userdata

def run(cmd, desc="", check=True):
    if desc:
        print(f"\n[⏳] {desc}")
    print(f"[CMD] {cmd}")
    p = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print((p.stdout or "").strip()[-1200:] if p.stdout else "[OK]")
    if check and p.returncode != 0:
        raise RuntimeError(f"命令失败: {cmd}")

def step(i, n, title):
    print(f"\n{'='*62}\n[{i}/{n}] {title}\n{'='*62}")

TOTAL = 7
COMFY_DIR = "/content/ComfyUI"
CUSTOM_NODES_DIR = f"{COMFY_DIR}/custom_nodes"
FRP_DIR = "/content/frp"
FRPC_BIN = f"{FRP_DIR}/frpc"

step(1, TOTAL, "挂载 Google Drive（可选）")
try:
    drive.mount('/content/drive')
except Exception as e:
    print(f"[WARN] 挂载失败（可忽略）: {e}")

step(2, TOTAL, "读取 Secrets")
try:
    VPS_IP = userdata.get("VPS_IP")
    FRP_TOKEN = userdata.get("FRP_TOKEN")
    print("✅ 已读取 VPS_IP / FRP_TOKEN")
except Exception:
    VPS_IP = "34.153.197.18"
    FRP_TOKEN = "Kaggle2026!"
    print("⚠️ 未读取到 Secrets，使用兜底值")

step(3, TOTAL, "安装系统依赖")
run("apt-get update -qq && apt-get install -y -qq git aria2", "安装 git/aria2")

step(4, TOTAL, "克隆 ComfyUI")
if not Path(COMFY_DIR).exists():
    run(f"git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR}", "clone ComfyUI")
else:
    print("ℹ️ ComfyUI 已存在，跳过")

step(5, TOTAL, "克隆 ComfyUI-Manager")
Path(CUSTOM_NODES_DIR).mkdir(parents=True, exist_ok=True)
manager_dir = Path(f"{CUSTOM_NODES_DIR}/ComfyUI-Manager")
if not manager_dir.exists():
    run(f"git clone --depth 1 https://github.com/ltdrdata/ComfyUI-Manager.git {manager_dir}", "clone Manager")
else:
    print("ℹ️ Manager 已存在，跳过")

step(6, TOTAL, "安装 Python 依赖")
run("python -m pip install -q -U pip setuptools wheel", "升级 pip")
req = Path(f"{COMFY_DIR}/requirements.txt")
if req.exists():
    run(f"python -m pip install -q --no-cache-dir -r {req}", "安装 ComfyUI requirements")

step(7, TOTAL, "部署 FRP")
Path(FRP_DIR).mkdir(parents=True, exist_ok=True)
if not Path(FRPC_BIN).exists():
    run(
        f"aria2c -c -x 16 -s 16 -k 1M -d {FRP_DIR} -o frp.tar.gz "
        "https://github.com/fatedier/frp/releases/download/v0.61.1/frp_0.61.1_linux_amd64.tar.gz",
        "下载 FRP"
    )
    run(f"tar -xzf {FRP_DIR}/frp.tar.gz -C {FRP_DIR} --strip-components=1", "解压 FRP")
    run(f"chmod +x {FRPC_BIN}", "设置执行权限")
else:
    print("ℹ️ frpc 已存在，跳过下载")

frpc_config = f"""
serverAddr = "{VPS_IP}"
serverPort = 7000
auth.method = "token"
auth.token = "{FRP_TOKEN}"

[[proxies]]
name = "comfyui_colab"
type = "tcp"
localIP = "127.0.0.1"
localPort = 8188
remotePort = 8188
""".strip()

with open(f"{FRP_DIR}/frpc.toml", "w", encoding="utf-8") as f:
    f.write(frpc_config + "\n")

print("\n✅ Cell1 完成")


[1/7] 挂载 Google Drive（可选）
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

[2/7] 读取 Secrets
✅ 已读取 VPS_IP / FRP_TOKEN

[3/7] 安装系统依赖

[⏳] 安装 git/aria2
[CMD] apt-get update -qq && apt-get install -y -qq git aria2
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 
(Reading database ... 5%
(Reading database ... 10%
(Reading database ... 15%
(Reading database ... 20%
(Reading database ... 25%
(Reading database ... 30%
(Reading database ... 35%
(Reading database ... 40%
(Reading database ... 45%
(Reading database ... 50%
(Reading database ... 55%
(Reading database ... 60%
(Reading database ... 65%
(Reading database ... 70%
(Reading database ... 75%
(Reading database ... 80%
(Reading database ... 85%
(Reading database ... 90%
(Reading database ... 95%
(Re

In [2]:
# ============================================================
# Cell 2 / 7 : 01_model_base_prepare_no_flux
# 基础模型准备（不包含 Flux 2D Assets 2026 V12）
# ============================================================
import os, glob, subprocess
from pathlib import Path

def run(cmd, desc="", check=True):
    if desc:
        print(f"\n[⏳] {desc}")
    p = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print((p.stdout or "").strip()[-1000:] if p.stdout else "[OK]")
    if check and p.returncode != 0:
        raise RuntimeError(f"命令失败: {cmd}")

def pbar(prefix, i, n, name=""):
    n = max(n, 1)
    d = int(i/n*24)
    print(f"{prefix} [{'#'*d}{'.'*(24-d)}] {i}/{n} {name}")

COMFY_DIR = "/content/ComfyUI"
MODELS_DIR = f"{COMFY_DIR}/models"
OUTPUT_DIR = f"{COMFY_DIR}/output"

# 不放 Flux 主模型；Flux 放到 Cell4 手工执行
LOCAL_GLOB_MODELS = [
    # ("checkpoints", "/content/dataset/animagine*.safetensors"),
    # ("loras", "/content/dataset/family-guy-style*.safetensors"),
]

EXTERNAL_URLS = [
    # ("checkpoints", "https://huggingface.co/.../model.safetensors", None),
]

print("[1/3] 创建目录并清理旧软链")
for d in ["checkpoints","loras","unet","clip","vae","clip_vision","controlnet","ipadapter","xlabs/ipadapters"]:
    Path(f"{MODELS_DIR}/{d}").mkdir(parents=True, exist_ok=True)
    run(f"find {MODELS_DIR}/{d} -type l -delete", f"清理 {d}", check=False)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("\n[2/3] 本地模型挂载（非 Flux 主模型）")
total = 0
for i, (subdir, pattern) in enumerate(LOCAL_GLOB_MODELS, 1):
    files = glob.glob(pattern, recursive=True)
    total += len(files)
    target = Path(f"{MODELS_DIR}/{subdir}")
    for src in files:
        dst = target / Path(src).name
        if dst.exists() or dst.is_symlink():
            dst.unlink()
        os.symlink(src, dst)
    pbar("挂载进度", i, len(LOCAL_GLOB_MODELS), f"{subdir}: {len(files)} files")
if not LOCAL_GLOB_MODELS:
    print("ℹ️ LOCAL_GLOB_MODELS 为空")
elif total == 0:
    print("⚠️ 未匹配到文件，请检查路径")

print("\n[3/3] 外链模型下载")
if EXTERNAL_URLS:
    for i, (subdir, url, filename) in enumerate(EXTERNAL_URLS, 1):
        target = Path(f"{MODELS_DIR}/{subdir}")
        fname = filename if filename else url.split("?")[0].split("/")[-1]
        run(f"aria2c -c -x 16 -s 16 -k 1M '{url}' -d '{target}' -o '{fname}'", f"下载 {fname}")
        pbar("下载进度", i, len(EXTERNAL_URLS), fname)
else:
    print("ℹ️ EXTERNAL_URLS 为空，跳过")

print(f"\n✅ Cell2 完成，输出目录：{OUTPUT_DIR}")

[1/3] 创建目录并清理旧软链

[⏳] 清理 checkpoints
[OK]

[⏳] 清理 loras
[OK]

[⏳] 清理 unet
[OK]

[⏳] 清理 clip
[OK]

[⏳] 清理 vae
[OK]

[⏳] 清理 clip_vision
[OK]

[⏳] 清理 controlnet
[OK]

[⏳] 清理 ipadapter
[OK]

[⏳] 清理 xlabs/ipadapters
[OK]

[2/3] 本地模型挂载（非 Flux 主模型）
ℹ️ LOCAL_GLOB_MODELS 为空

[3/3] 外链模型下载
ℹ️ EXTERNAL_URLS 为空，跳过

✅ Cell2 完成，输出目录：/content/ComfyUI/output


In [3]:
# ============================================================
# Cell 3 / 7 : 02_flux_text_encoder_models
# 下载 ae / clip_l / t5xxl（按需）
# ============================================================
import os, subprocess
from pathlib import Path
from google.colab import userdata

def run(cmd, desc="", check=True):
    if desc:
        print(f"\n[⏳] {desc}")
    print(f"[CMD] {cmd}")
    p = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print((p.stdout or "").strip()[-1200:] if p.stdout else "[OK]")
    if check and p.returncode != 0:
        raise RuntimeError(f"命令失败: {cmd}")

print("[1/4] 读取 HF_TOKEN")
HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
print("✅ HF_TOKEN 已加载")

print("[2/4] 安装 aria2")
run("apt-get update -qq && apt-get install -y -qq aria2", "安装 aria2")

print("[3/4] 创建目录")
Path("/content/ComfyUI/models/vae").mkdir(parents=True, exist_ok=True)
Path("/content/ComfyUI/models/clip").mkdir(parents=True, exist_ok=True)

downloads = [
    ("/content/ComfyUI/models/vae", "ae.safetensors",
     "https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/ae.safetensors"),
    ("/content/ComfyUI/models/clip", "clip_l.safetensors",
     "https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors"),
    ("/content/ComfyUI/models/clip", "t5xxl_fp16.safetensors",
     "https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp16.safetensors"),
]

print("[4/4] 下载模型")
for i, (dest, fname, url) in enumerate(downloads, 1):
    print(f"[{i}/{len(downloads)}] {fname}")
    run(
        f'aria2c --header="Authorization: Bearer $HF_TOKEN" -c -x 16 -s 16 -k 1M "{url}" -d "{dest}" -o "{fname}"',
        f"下载 {fname}"
    )

print("\n✅ Cell3 完成")

[1/4] 读取 HF_TOKEN
✅ HF_TOKEN 已加载
[2/4] 安装 aria2

[⏳] 安装 aria2
[CMD] apt-get update -qq && apt-get install -y -qq aria2
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
[3/4] 创建目录
[4/4] 下载模型
[1/3] ae.safetensors

[⏳] 下载 ae.safetensors
[CMD] aria2c --header="Authorization: Bearer $HF_TOKEN" -c -x 16 -s 16 -k 1M "https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/ae.safetensors" -d "/content/ComfyUI/models/vae" -o "ae.safetensors"
27%27ae.safetensors%3B+filename%3D%22ae.safetensors%22%3B&x-amz-checksum-mode=ENABLED&x-id=GetObject&Expires=1774284737&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc3NDI4NDczN319LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2FzLWJyaWRnZS54ZXRodWIuaGYuY28veGV0LWJyaWRnZS11cy82NmFhOTc0ZDFmODNiMjEwYWU3Zjc0YWUvZjczZWVjZjdjNDY5ZmY0NDI1MjNkYzcxMmNjMTYxZDYzMWRmMDcxYmY0ZDlkNzkzNDk0ZmJmMDB

In [4]:
# cell4: 直接从 HF 下载官方 FLUX.1 [dev] 原版权重 (需申请权限 + HF_TOKEN)
import os, subprocess
from pathlib import Path
from google.colab import userdata

# 官方 FLUX.1-dev 完整版 Unet 的真实直链 (约 23.8 GB)
HF_URL = "https://huggingface.co/black-forest-labs/FLUX.1-dev/resolve/main/flux1-dev.safetensors"
FILE_NAME = "flux1-dev.safetensors"

COMFY_DIR = Path("/content/ComfyUI")
UNET_DIR = COMFY_DIR / "models/unet"
CKPT_DIR = COMFY_DIR / "models/checkpoints"
TARGET_PATH = UNET_DIR / FILE_NAME

def run(cmd, desc="", check=True):
    if desc:
        print(f"\n[⏳] {desc}")
    # 让 aria2 的实时进度条直接输出到屏幕
    p = subprocess.run(cmd, shell=True)
    if check and p.returncode != 0:
        raise RuntimeError(f"❌ 命令失败！请检查：1.网络是否正常 2.HF_TOKEN是否正确 3.是否已在HF官网同意了该模型的协议")

print("\n================ [1/4] 获取 Hugging Face 凭证 ================")
try:
    hf_token = userdata.get("HF_TOKEN")
    print("✅ 成功从 Colab Secrets 获取 HF_TOKEN")
except userdata.SecretNotFoundError:
    raise RuntimeError("❌ 致命错误：未找到 HF_TOKEN！请在 Colab 左侧的 🔑 Secrets 中添加变量。")

print("\n================ [2/4] 环境准备 ================")
# 安装 aria2 (如果已安装会自动跳过，加了 -q 静默输出)
subprocess.run("apt-get update -qq && apt-get install -y -qq aria2", shell=True)
UNET_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n================ [3/4] aria2 满速直连下载 {FILE_NAME} ================")
# 核心下载命令：注入 HF_TOKEN，使用 16 线程满速下载
cmd = (
    f'aria2c -c -x 16 -s 16 -k 1M '
    f'--summary-interval=1 --download-result=full '
    f'--header="Authorization: Bearer {hf_token}" '
    f'--out "{FILE_NAME}" -d "{UNET_DIR}" '
    f'"{HF_URL}"'
)
run(cmd, "正在携带 Token 拉取官方模型 (这可能需要几分钟，请观察下方进度条)...")

if not TARGET_PATH.exists():
    raise RuntimeError("❌ 下载失败，未找到模型文件。")

print("\n================ [4/4] 挂载到 ComfyUI ================")
# 官方标准工作流通常把这个文件当做 unet 节点加载
# 但为了兼容性，我们在 checkpoints 目录也做个软链接
ckpt_link = CKPT_DIR / FILE_NAME
if ckpt_link.exists() or ckpt_link.is_symlink():
    try: ckpt_link.unlink()
    except: pass
os.symlink(str(TARGET_PATH), str(ckpt_link))

print(f"✅ 模型本体已就绪: {TARGET_PATH}")
print(f"✅ 快捷软链接已建立: {ckpt_link} -> {TARGET_PATH}")
print("\n🎉 完美！官方原汁原味的 FLUX.1 [dev] 已经下载完毕！")


================ [1/4] 获取 Hugging Face 凭证 ================
✅ 成功从 Colab Secrets 获取 HF_TOKEN

================ [2/4] 环境准备 ================

================ [3/4] aria2 满速直连下载 flux1-dev.safetensors ================

[⏳] 正在携带 Token 拉取官方模型 (这可能需要几分钟，请观察下方进度条)...

================ [4/4] 挂载到 ComfyUI ================
✅ 模型本体已就绪: /content/ComfyUI/models/unet/flux1-dev.safetensors
✅ 快捷软链接已建立: /content/ComfyUI/models/checkpoints/flux1-dev.safetensors -> /content/ComfyUI/models/unet/flux1-dev.safetensors

🎉 完美！官方原汁原味的 FLUX.1 [dev] 已经下载完毕！


In [5]:
# ============================================================
# Cell 5 / 7 : 04_plugin_base_install
# 基础插件安装（常用固定插件）
# ============================================================
import subprocess
from pathlib import Path

def run(cmd, desc="", check=True):
    if desc:
        print(f"\n[⏳] {desc}")
    p = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print((p.stdout or "").strip()[-1000:] if p.stdout else "[OK]")
    if check and p.returncode != 0:
        raise RuntimeError(f"命令失败: {cmd}")

COMFY_DIR = "/content/ComfyUI"
CUSTOM_NODES_DIR = f"{COMFY_DIR}/custom_nodes"
Path(CUSTOM_NODES_DIR).mkdir(parents=True, exist_ok=True)

BASE_PLUGINS = [
    "https://github.com/cubiq/ComfyUI_IPAdapter_plus.git",
    "https://github.com/Fannovel16/comfyui_controlnet_aux.git",
    "https://github.com/rgthree/rgthree-comfy.git",
]

for i, url in enumerate(BASE_PLUGINS, 1):
    name = url.rstrip("/").split("/")[-1].replace(".git", "")
    dst = Path(f"{CUSTOM_NODES_DIR}/{name}")
    print(f"\n[{i}/{len(BASE_PLUGINS)}] {name}")
    if not dst.exists():
        run(f"git clone --depth 1 {url} {dst}", f"克隆 {name}")
    else:
        run(f"cd {dst} && git pull -q", f"更新 {name}", check=False)

    req = dst / "requirements.txt"
    if req.exists():
        run(f"python -m pip install -q --no-cache-dir -r {req}", f"安装依赖 {name}")

print("\n✅ Cell5 完成")


[1/3] ComfyUI_IPAdapter_plus

[⏳] 克隆 ComfyUI_IPAdapter_plus
Cloning into '/content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus'...

[2/3] comfyui_controlnet_aux

[⏳] 克隆 comfyui_controlnet_aux
Cloning into '/content/ComfyUI/custom_nodes/comfyui_controlnet_aux'...

[⏳] 安装依赖 comfyui_controlnet_aux
[OK]

[3/3] rgthree-comfy

[⏳] 克隆 rgthree-comfy
Cloning into '/content/ComfyUI/custom_nodes/rgthree-comfy'...

[⏳] 安装依赖 rgthree-comfy
[OK]

✅ Cell5 完成


In [6]:
# ============================================================
# Cell 6 / 7 : 05_expansion_center
# 扩展中心（新增节点 + 新模型），不包含 Flux 2D Assets 2026 V12
# ============================================================
import os, glob, subprocess
from pathlib import Path

def run(cmd, desc="", check=True):
    if desc:
        print(f"\n[⏳] {desc}")
    print(f"[CMD] {cmd}")
    p = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print((p.stdout or "").strip()[-1200:] if p.stdout else "[OK]")
    if check and p.returncode != 0:
        raise RuntimeError(f"命令失败: {cmd}")

def pbar(prefix, i, n, name=""):
    n = max(n, 1)
    d = int(i / n * 24)
    print(f"{prefix} [{'#'*d}{'.'*(24-d)}] {i}/{n} {name}")

COMFY_DIR = "/content/ComfyUI"
CUSTOM_NODES_DIR = f"{COMFY_DIR}/custom_nodes"
MODELS_DIR = f"{COMFY_DIR}/models"
Path(CUSTOM_NODES_DIR).mkdir(parents=True, exist_ok=True)
Path(MODELS_DIR).mkdir(parents=True, exist_ok=True)
Path("/content/dataset").mkdir(parents=True, exist_ok=True)

# ===== 你只改这里 =====
EXPANSION_MODE = True

NEW_CUSTOM_NODES = [
    # "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git",
]

NEW_MODELS_URL = [
    # ("checkpoints", "https://huggingface.co/.../model.safetensors", None),
]

NEW_MODELS_HF = [
    # ("XLabs-AI/flux-ip-adapter", "ip_adapter.safetensors", "xlabs/ipadapters"),
]

NEW_MODELS_GLOB = [
    # ("loras", "/content/dataset/**/*.safetensors"),
]
# ======================

print("[0/4] Manager 显隐切换")
manager_dir = Path(f"{CUSTOM_NODES_DIR}/ComfyUI-Manager")
hidden_dir = Path(f"{COMFY_DIR}/ComfyUI-Manager_hidden")
if EXPANSION_MODE:
    if hidden_dir.exists() and not manager_dir.exists():
        hidden_dir.rename(manager_dir)
    print("✅ Manager 已开启（扩展模式）")
else:
    if manager_dir.exists() and not hidden_dir.exists():
        manager_dir.rename(hidden_dir)
    print("⚡ Manager 已隐藏（极速模式）")

print("\n[1/4] 新增节点")
for i, url in enumerate(NEW_CUSTOM_NODES, 1):
    name = url.rstrip("/").split("/")[-1].replace(".git", "")
    dst = Path(f"{CUSTOM_NODES_DIR}/{name}")
    pbar("节点进度", i, len(NEW_CUSTOM_NODES), name)
    if not dst.exists():
        run(f"git clone --depth 1 {url} {dst}", f"克隆 {name}")
    else:
        run(f"cd {dst} && git pull -q", f"更新 {name}", check=False)
    req = dst / "requirements.txt"
    if req.exists():
        run(f"python -m pip install -q --no-cache-dir -r {req}", f"安装依赖 {name}")
if not NEW_CUSTOM_NODES:
    print("ℹ️ 无新增节点")

print("\n[2/4] URL 模型下载")
run("apt-get update -qq && apt-get install -y -qq aria2", "安装 aria2")
for i, (subdir, url, fname) in enumerate(NEW_MODELS_URL, 1):
    target = Path(f"{MODELS_DIR}/{subdir}")
    target.mkdir(parents=True, exist_ok=True)
    name = fname if fname else url.split("?")[0].split("/")[-1]
    pbar("URL进度", i, len(NEW_MODELS_URL), name)
    run(f"aria2c -c -x 16 -s 16 -k 1M '{url}' -d '{target}' -o '{name}'", f"下载 {name}")
if not NEW_MODELS_URL:
    print("ℹ️ 无 URL 模型")

print("\n[3/4] HF 模型下载")
if NEW_MODELS_HF:
    run("python -m pip install -q -U huggingface_hub", "安装 huggingface_hub")
    from huggingface_hub import hf_hub_download
    for i, (repo_id, filename, subdir) in enumerate(NEW_MODELS_HF, 1):
        pbar("HF进度", i, len(NEW_MODELS_HF), filename)
        local = hf_hub_download(repo_id=repo_id, filename=filename, local_dir="/content/dataset")
        target = Path(f"{MODELS_DIR}/{subdir}")
        target.mkdir(parents=True, exist_ok=True)
        dst = target / Path(filename).name
        if dst.exists() or dst.is_symlink():
            dst.unlink()
        os.symlink(local, dst)
        print(f"✅ {dst} -> {local}")
else:
    print("ℹ️ 无 HF 模型")

print("\n[4/4] GLOB 本地挂载")
for i, (subdir, pattern) in enumerate(NEW_MODELS_GLOB, 1):
    target = Path(f"{MODELS_DIR}/{subdir}")
    target.mkdir(parents=True, exist_ok=True)
    files = glob.glob(pattern, recursive=True)
    pbar("挂载进度", i, len(NEW_MODELS_GLOB), f"{subdir}: {len(files)} files")
    for src in files:
        dst = target / Path(src).name
        if dst.exists() or dst.is_symlink():
            dst.unlink()
        os.symlink(src, dst)
if not NEW_MODELS_GLOB:
    print("ℹ️ 无 GLOB 挂载任务")

print("\n✅ Cell6 完成（扩展任务已应用）")

[0/4] Manager 显隐切换
✅ Manager 已开启（扩展模式）

[1/4] 新增节点
ℹ️ 无新增节点

[2/4] URL 模型下载

[⏳] 安装 aria2
[CMD] apt-get update -qq && apt-get install -y -qq aria2
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
ℹ️ 无 URL 模型

[3/4] HF 模型下载
ℹ️ 无 HF 模型

[4/4] GLOB 本地挂载
ℹ️ 无 GLOB 挂载任务

✅ Cell6 完成（扩展任务已应用）


In [8]:
# ============================================================
# Cell 7 / 7 : 06_start_runtime_only
# 日常只运行这个：启动 ComfyUI + FRP
# ============================================================
import os, time, socket, subprocess
from pathlib import Path

COMFY_DIR = "/content/ComfyUI"
FRP_DIR = "/content/frp"
FRPC_BIN = f"{FRP_DIR}/frpc"
VPS_IP = "34.153.197.18"

ENABLE_MANAGER = False  # 日常 False；需要 UI 装节点时 True

def wait_port(host, port, timeout=120):
    start = time.time()
    while time.time() - start < timeout:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex((host, port)) == 0:
                return True
        time.sleep(1)
    return False

print("[1/6] 启动前自检")
need = [
    "/content/ComfyUI/main.py",
    "/content/frp/frpc",
    "/content/frp/frpc.toml",
]
for p in need:
    if not Path(p).exists():
        raise FileNotFoundError(f"缺少文件: {p}，请先运行 Cell1")
print("✅ 基础文件存在")

print("[2/6] 清理旧进程")
os.system("pkill -f 'python.*main.py' || true")
os.system("pkill -f frpc || true")
time.sleep(1)

print("[3/6] Manager 状态切换")
manager_dir = Path(f"{COMFY_DIR}/custom_nodes/ComfyUI-Manager")
hidden_dir = Path(f"{COMFY_DIR}/ComfyUI-Manager_hidden")
if ENABLE_MANAGER:
    if hidden_dir.exists() and not manager_dir.exists():
        hidden_dir.rename(manager_dir)
else:
    if manager_dir.exists() and not hidden_dir.exists():
        manager_dir.rename(hidden_dir)

print("[4/6] 启动 ComfyUI")
comfy = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188", "--normalvram"],
    cwd=COMFY_DIR
)

print("[5/6] 等待端口 8188")
if not wait_port("127.0.0.1", 8188, timeout=120):
    raise RuntimeError("ComfyUI 启动超时，请查看日志")

print("[6/6] 启动 FRP")
frp = subprocess.Popen([FRPC_BIN, "-c", f"{FRP_DIR}/frpc.toml"])

print("\n" + "="*56)
print("🎉 ComfyUI 已启动")
print(f"🌐 访问: http://{VPS_IP}:8188")
print(f"🧩 Manager: {'开启' if ENABLE_MANAGER else '隐藏'}")
print("="*56 + "\n")

comfy.wait()

[1/6] 启动前自检


FileNotFoundError: 缺少文件: /content/ComfyUI/main.py，请先运行 Cell1